# XGBoost Architecture-Lock Notebook

## Main question

Can a tabular XGBoost model be locked into one defensible preset using the same thesis data pipeline as the LSTM work, while staying comparable across cumulative horizons, per-building runs, and a pooled global run, and optionally extended with oracle look-ahead weather?

This notebook is intentionally about **locking the XGBoost setup**, not about making final thesis claims.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd()
if cwd.name == 'thesis-project':
    PROJECT_ROOT = cwd.resolve()
elif (cwd / 'thesis-project').exists():
    PROJECT_ROOT = (cwd / 'thesis-project').resolve()
else:
    PROJECT_ROOT = cwd.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib.util
import subprocess

if importlib.util.find_spec("xgboost") is None:
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "xgboost"],
        )
        import importlib
        importlib.invalidate_caches()
        display(Markdown("> Installed `xgboost` for this kernel (`python -m pip install xgboost`)."))
    except subprocess.CalledProcessError:
        pass

from utils.xgboost_architecture_lock import (
    NotebookConfig,
    artifact_paths,
    build_building_catalog_df,
    build_mode_catalog_df,
    build_summary_artifacts,
    build_target_preview,
    notebook_runtime_note,
    run_full_matrix,
    run_preset_audit,
    select_best_preset,
    xgboost_available,
)

MAIN_BUILDINGS = ('U05', 'U06', 'LIB', 'U02B', 'SOC', 'U03')
MAIN_HORIZONS = (8, 12, 16, 20, 24, 36)
AUDIT_HORIZONS = (8, 12, 24, 36)
USE_FUTURE_WEATHER = True

# Keep the overnight run aligned with the thesis-facing long-horizon cohort.
config = NotebookConfig(
    buildings=MAIN_BUILDINGS,
    horizons=MAIN_HORIZONS,
    audit_buildings=('U05', 'U06'),
    audit_horizons=AUDIT_HORIZONS,
    use_future_weather=USE_FUTURE_WEATHER,
)
paths = artifact_paths(config)
XGBOOST_AVAILABLE = xgboost_available()

display(Markdown('```text\n' + notebook_runtime_note(config) + '\n```'))
if not XGBOOST_AVAILABLE:
    _mac_help = (
        ' On macOS, if the native library fails (`libomp.dylib`), install OpenMP with the Homebrew '
        'that matches your machine: `/opt/homebrew/bin/brew install libomp` (Apple Silicon) or '
        '`/usr/local/bin/brew install libomp` (Intel).'
        if sys.platform == 'darwin'
        else ''
    )
    display(Markdown(
        '> `xgboost` is missing or failed to load in this kernel. '
        'Try `python -m pip install xgboost`, restart the kernel, and rerun this cell.'
        + _mac_help
    ))

```text
Selected buildings: U05, U06, LIB, U02B, SOC, U03
Audit buildings: U05, U06
Horizons: [8, 12, 16, 20, 24, 36]
Audit horizons: [8, 12, 24, 36]
Modes: ['M0', 'M1', 'M2', 'M3']
Oracle future weather: True
Planned audit rows: 120
Planned full-matrix rows: 168
xgboost available: True
Results dir: /Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/results
```

In [2]:
# Setup is handled in the import cell above; keep this cell as a no-op.

## Why this notebook exists

The role of this notebook is to define a fair XGBoost comparison frame before interpretability and model-family claims expand further.

Fairness rules used here:

- same thesis feature pipeline as the LSTM work
- same chronological split direction with train through the end of 2023 and future evaluation in 2024
- same cumulative target family across the selected horizon set
- same `M0-M3` mode framing, with optional horizon-specific future-weather columns appended as a separate rerun
- same main reporting pair: `RMSE + WAPE`
- per-building and pooled-global XGBoost are both included, but static building descriptors are used only in the pooled-global track

In [3]:
# Resume-friendly overnight defaults: reruns continue from the saved CSV artifacts.
RUN_PRESET_AUDIT = True
RUN_FULL_MATRIX = True
REBUILD_SUMMARIES = True

audit_df = None
audit_summary_df = None
selected_preset = None
per_df = None
global_df = None
summary_df = None
matched_df = None

if not RUN_PRESET_AUDIT and paths['audit'].exists():
    audit_df = pd.read_csv(paths['audit'])
    selected_preset, audit_summary_df = select_best_preset(audit_df, config)

display(pd.DataFrame([
    {'flag': 'RUN_PRESET_AUDIT', 'value': RUN_PRESET_AUDIT},
    {'flag': 'RUN_FULL_MATRIX', 'value': RUN_FULL_MATRIX},
    {'flag': 'REBUILD_SUMMARIES', 'value': REBUILD_SUMMARIES},
]))

,flag,value
0,RUN_PRESET_AUDIT,True
1,RUN_FULL_MATRIX,True
2,REBUILD_SUMMARIES,True


## Data sources and fairness rules

Per-building models are trained from `setA`, which keeps the track dynamic and close to the lean LSTM feature framing.

The pooled global model is trained from `setB`, but only to access safe static building context. The notebook deliberately avoids the `stat_train_*` family and similar target-derived summary columns in order to reduce leakage risk.

This means:

- per-building XGBoost tests tabular learning from dynamic operational signals
- pooled global XGBoost tests whether safe building context helps a single model generalize across buildings

In [4]:
building_catalog_df = build_building_catalog_df(config)
mode_catalog_df = build_mode_catalog_df(config)
artifact_df = pd.DataFrame([
    {'artifact': key, 'path': str(path)}
    for key, path in paths.items()
])

display(building_catalog_df)
display(mode_catalog_df)
display(artifact_df)

,building,rows_setA,rows_setB,datetime_start,datetime_end,heating_share_setA_pct,global_static_complete,target_cum_h8_nonnull_train,target_cum_h8_nonnull_test,target_cum_h12_nonnull_train,target_cum_h12_nonnull_test,target_cum_h16_nonnull_train,target_cum_h16_nonnull_test,target_cum_h20_nonnull_train,target_cum_h20_nonnull_test,target_cum_h24_nonnull_train,target_cum_h24_nonnull_test,target_cum_h36_nonnull_train,target_cum_h36_nonnull_test
0,LIB,22954,22954,2022-05-20 14:00:00,2024-12-31 23:00:00,72.741134,True,14163,8777,14159,8773,14155,8769,14151,8765,14147,8761,14135,8749
1,SOC,22954,22954,2022-05-20 14:00:00,2024-12-31 23:00:00,72.741134,True,14163,8777,14159,8773,14155,8769,14151,8765,14147,8761,14135,8749
2,U02B,22925,22925,2022-05-20 14:00:00,2024-12-30 18:00:00,72.706652,True,14163,8748,14159,8744,14155,8740,14151,8736,14147,8732,14135,8720
3,U03,22925,22925,2022-05-20 14:00:00,2024-12-30 18:00:00,72.706652,True,14163,8748,14159,8744,14155,8740,14151,8736,14147,8732,14135,8720
4,U05,25206,25206,2022-02-14 13:00:00,2024-12-30 18:00:00,75.124970,True,16444,8748,16440,8744,16436,8740,16432,8736,16428,8732,16416,8720
5,U06,25206,25206,2022-02-14 13:00:00,2024-12-30 18:00:00,75.124970,True,16444,8748,16440,8744,16436,8740,16432,8736,16428,8732,16416,8720


,mode,dynamic_feature_count,dynamic_feature_list,future_weather_feature_count,future_weather_feature_list,global_static_feature_count,global_static_feature_list
0,M0,8,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_...",4,"feat_fw_temp_mean_h08, feat_fw_temp_min_h08, f...",28,"stat_n_points, stat_n_vent_points, stat_n_heat..."
1,M1,9,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_...",4,"feat_fw_temp_mean_h08, feat_fw_temp_min_h08, f...",28,"stat_n_points, stat_n_vent_points, stat_n_heat..."
2,M2,14,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_...",4,"feat_fw_temp_mean_h08, feat_fw_temp_min_h08, f...",28,"stat_n_points, stat_n_vent_points, stat_n_heat..."
3,M3,15,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_...",4,"feat_fw_temp_mean_h08, feat_fw_temp_min_h08, f...",28,"stat_n_points, stat_n_vent_points, stat_n_heat..."


,artifact,path
0,audit,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
1,per_building,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
2,global,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
3,summary,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
4,matched,/Users/mihkeluutar/Documents/TalTech/Thesis/th...


## Target construction

This notebook uses the same cumulative-target construction already used in the recent cumulative LSTM notebooks.

The key comparability rule is the issue-time truncation:

- for horizon `H`, the final valid train issue time is `TRAIN_END - (H - 1)` hours
- validation rows come after the fit rows
- test rows begin in 2024 and remain strictly future-facing

In [5]:
target_preview_df = build_target_preview(config, building='U05', horizon_h=8)
display(target_preview_df)

,datetime,heat_kwh,target_cum_h8,feat_fw_temp_mean_h08,feat_fw_temp_min_h08,feat_fw_temp_end_h08,feat_fw_rh_mean_h08,train_issue_end
0,2022-02-14 13:00:00,2.000000,15.733333,2.42500,1.85,3.30,81.375,2023-12-31 16:00:00
1,2022-02-14 14:00:00,1.866667,15.676190,2.59375,1.85,3.40,79.125,2023-12-31 16:00:00
2,2022-02-14 15:00:00,2.000000,15.480952,2.80000,1.85,3.50,76.625,2023-12-31 16:00:00
3,2023-12-31 14:00:00,92.900000,802.733333,-6.37500,-8.05,-8.05,83.125,2023-12-31 16:00:00
4,2023-12-31 15:00:00,94.166667,808.500000,-6.55625,-8.05,-7.80,84.250,2023-12-31 16:00:00
5,2023-12-31 16:00:00,106.000000,801.666667,-6.75000,-8.05,-7.50,85.750,2023-12-31 16:00:00


## Feature modes

The notebook keeps the thesis mode framing exactly as the long-horizon LSTM work:

- `M0`: lean temporal core
- `M1`: `M0 +` weather / temperature memory
- `M2`: `M0 +` system / inertia proxies
- `M3`: `M0 +` both

This keeps the XGBoost comparison interpretable as a model-family contrast rather than a drifting feature-engineering exercise.

In [6]:
display(mode_catalog_df)

,mode,dynamic_feature_count,dynamic_feature_list,future_weather_feature_count,future_weather_feature_list,global_static_feature_count,global_static_feature_list
0,M0,8,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_...",4,"feat_fw_temp_mean_h08, feat_fw_temp_min_h08, f...",28,"stat_n_points, stat_n_vent_points, stat_n_heat..."
1,M1,9,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_...",4,"feat_fw_temp_mean_h08, feat_fw_temp_min_h08, f...",28,"stat_n_points, stat_n_vent_points, stat_n_heat..."
2,M2,14,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_...",4,"feat_fw_temp_mean_h08, feat_fw_temp_min_h08, f...",28,"stat_n_points, stat_n_vent_points, stat_n_heat..."
3,M3,15,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_...",4,"feat_fw_temp_mean_h08, feat_fw_temp_min_h08, f...",28,"stat_n_points, stat_n_vent_points, stat_n_heat..."


## XGBoost preset audit

The audit stage is intentionally small and representative:

- buildings: `U05`, `U06`
- modes: `M0`, `M3`
- horizons: `8`, `12`, `24`, `36`
- scopes: per-building and pooled-global

Selection rule:

1. lowest mean validation `WAPE`
2. lower mean validation `RMSE` as tie-break
3. simplest preset within a small `WAPE` margin

In [7]:
if RUN_PRESET_AUDIT:
    if not XGBOOST_AVAILABLE:
        raise RuntimeError('Set up xgboost first, then rerun the preset-audit cell.')
    audit_df, selected_preset, audit_summary_df = run_preset_audit(config)
elif paths['audit'].exists():
    if audit_df is None or selected_preset is None:
        audit_df = pd.read_csv(paths['audit'])
        selected_preset, audit_summary_df = select_best_preset(audit_df, config)
else:
    display(Markdown(
        f'> No audit CSV at `{paths["audit"]}`. Rerun the **flags** cell (it sets `RUN_PRESET_AUDIT` when the file is missing), '
        'then this cell—or run the **full-matrix** cell, which can fit the audit when `RUN_PRESET_AUDIT` is `True`.'
    ))

if audit_df is not None:
    display(audit_df.head(20))
if audit_summary_df is not None:
    display(audit_summary_df)
if selected_preset is not None:
    display(pd.DataFrame([selected_preset]))

Preset audit scope: 2 buildings x 2 modes x 4 horizons x 5 presets
[  1/120] audit | P01_md3_lr003_mc5 | per_building | U05 | M0 | h= 8 | status=ok | val_wape=15.018
[  2/120] audit | P01_md3_lr003_mc5 | per_building | U06 | M0 | h= 8 | status=ok | val_wape=10.169
[  3/120] audit | P01_md3_lr003_mc5 | global | GLOBAL_AUDIT | M0 | h= 8 | status=ok | val_wape=11.914
[  4/120] audit | P01_md3_lr003_mc5 | per_building | U05 | M0 | h=12 | status=ok | val_wape=16.374
[  5/120] audit | P01_md3_lr003_mc5 | per_building | U06 | M0 | h=12 | status=ok | val_wape=12.223
[  6/120] audit | P01_md3_lr003_mc5 | global | GLOBAL_AUDIT | M0 | h=12 | status=ok | val_wape=13.478
[  7/120] audit | P01_md3_lr003_mc5 | per_building | U05 | M0 | h=24 | status=ok | val_wape=17.468
[  8/120] audit | P01_md3_lr003_mc5 | per_building | U06 | M0 | h=24 | status=ok | val_wape=12.044
[  9/120] audit | P01_md3_lr003_mc5 | global | GLOBAL_AUDIT | M0 | h=24 | status=ok | val_wape=14.019
[ 10/120] audit | P01_md3_lr003_m

,scope,building_or_global,mode,horizon_h,target_name,preset_id,n_features,feature_list,n_train_rows,n_val_rows,...,rmse,mae,wape_pct,r2,baseline_wape_pct,delta_wape_vs_baseline,n_test_eval_rows,max_depth,learning_rate,min_child_weight
0,global,GLOBAL_AUDIT,M0,8,target_cum_h8,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",29599,3289,...,133.662869,81.665902,14.339065,0.908275,17.820101,-3.481036,12702,3,0.03,5
1,global,GLOBAL_AUDIT,M0,12,target_cum_h12,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",29592,3288,...,212.341976,133.260976,15.615778,0.892305,23.241447,-7.625669,12694,3,0.03,5
2,global,GLOBAL_AUDIT,M0,24,target_cum_h24,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",29570,3286,...,435.989963,274.053771,16.100099,0.877369,24.494085,-8.393985,12670,3,0.03,5
3,global,GLOBAL_AUDIT,M0,36,target_cum_h36,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",29549,3283,...,665.502857,417.819394,16.378355,0.870954,25.988855,-9.610501,12646,3,0.03,5
4,global,GLOBAL_AUDIT,M3,8,target_cum_h8,P01_md3_lr003_mc5,47,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",29599,3289,...,134.641637,82.072994,14.410542,0.906927,17.820101,-3.409558,12702,3,0.03,5
5,global,GLOBAL_AUDIT,M3,12,target_cum_h12,P01_md3_lr003_mc5,47,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",29592,3288,...,214.485287,133.566481,15.651578,0.890120,23.241447,-7.589869,12694,3,0.03,5
6,global,GLOBAL_AUDIT,M3,24,target_cum_h24,P01_md3_lr003_mc5,47,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",29570,3286,...,429.326728,269.045245,15.805859,0.881089,24.494085,-8.688226,12670,3,0.03,5
7,global,GLOBAL_AUDIT,M3,36,target_cum_h36,P01_md3_lr003_mc5,47,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",29549,3283,...,646.091807,405.559037,15.897754,0.878372,25.988855,-10.091102,12646,3,0.03,5
8,per_building,U05,M0,8,target_cum_h8,P01_md3_lr003_mc5,12,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",14800,1644,...,161.441815,101.122963,19.757257,0.869765,26.168225,-6.410968,6351,3,0.03,5
9,per_building,U05,M0,12,target_cum_h12,P01_md3_lr003_mc5,12,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",14796,1644,...,256.286598,163.437081,21.300144,0.842629,35.046333,-13.746189,6347,3,0.03,5


,preset_id,n_runs,mean_val_wape,mean_val_rmse,mean_best_iteration,max_depth,learning_rate,min_child_weight
0,P01_md3_lr003_mc5,24,13.445528,358.694773,1999.0,3,0.03,5
1,P02_md3_lr005_mc1,24,13.520456,359.488679,1999.0,3,0.05,1
2,P03_md5_lr005_mc5,24,13.844895,366.877689,1999.0,5,0.05,5
3,P04_md5_lr010_mc1,24,13.934281,368.924137,1999.0,5,0.10,1
4,P05_md7_lr005_mc5,24,14.160558,375.843562,1999.0,7,0.05,5


,preset_id,max_depth,learning_rate,min_child_weight
0,P01_md3_lr003_mc5,3,0.03,5


## Frozen-preset per-building and pooled-global runs

After the audit stage picks one preset, the notebook runs the full matrix:

- per-building: thesis six-building cohort × `M0-M3` × cumulative horizons `8-36`
- pooled global: one model per `mode × horizon` across the same six-building cohort

The goal here is not to over-tune XGBoost separately per building, but to see what one locked setup does under a fair, repeated comparison frame.

In [8]:
if RUN_FULL_MATRIX:
    if not XGBOOST_AVAILABLE:
        raise RuntimeError('Set up xgboost first, then rerun the full-matrix cell.')
    if selected_preset is None and paths['audit'].exists():
        audit_df = pd.read_csv(paths['audit'])
        selected_preset, audit_summary_df = select_best_preset(audit_df, config)
    if selected_preset is None and RUN_PRESET_AUDIT:
        audit_df, selected_preset, audit_summary_df = run_preset_audit(config)
    if selected_preset is None:
        raise RuntimeError(
            'No preset: run the preset-audit code cell above, or set RUN_PRESET_AUDIT = True and rerun the flags cell. '
            f'Expected audit CSV: {paths["audit"]}'
        )
    per_df, global_df, selected_preset = run_full_matrix(config, preset=selected_preset)
else:
    if paths['per_building'].exists():
        per_df = pd.read_csv(paths['per_building'])
    if paths['global'].exists():
        global_df = pd.read_csv(paths['global'])

if per_df is not None:
    display(per_df.head(20))
if global_df is not None:
    display(global_df.head(20))

Full matrix scope: 6 buildings x 4 modes x 6 horizons
[  1/144] per_building | U05 | M0 | h= 8 | status=ok | wape=19.757
[  2/144] per_building | U06 | M0 | h= 8 | status=ok | wape=11.638
[  3/144] per_building | LIB | M0 | h= 8 | status=ok | wape=12.818
[  4/144] per_building | U02B | M0 | h= 8 | status=ok | wape=20.568
[  5/144] per_building | SOC | M0 | h= 8 | status=ok | wape=14.495
[  6/144] per_building | U03 | M0 | h= 8 | status=ok | wape=12.773
[  7/144] per_building | U05 | M0 | h=12 | status=ok | wape=21.300
[  8/144] per_building | U06 | M0 | h=12 | status=ok | wape=12.987
[  9/144] per_building | LIB | M0 | h=12 | status=ok | wape=14.457
[ 10/144] per_building | U02B | M0 | h=12 | status=ok | wape=21.830
[ 11/144] per_building | SOC | M0 | h=12 | status=ok | wape=16.778
[ 12/144] per_building | U03 | M0 | h=12 | status=ok | wape=14.475
[ 13/144] per_building | U05 | M0 | h=16 | status=ok | wape=22.050
[ 14/144] per_building | U06 | M0 | h=16 | status=ok | wape=13.699
[ 15/1

,scope,building_or_global,mode,horizon_h,target_name,preset_id,n_features,feature_list,n_train_rows,n_val_rows,...,best_iteration,val_wape,val_rmse,rmse,mae,wape_pct,r2,baseline_wape_pct,delta_wape_vs_baseline,n_test_eval_rows
0,per_building,LIB,M0,8,target_cum_h8,P01_md3_lr003_mc5,12,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12747,1416,...,1999,7.557135,94.504293,121.896232,79.878367,12.817594,0.930958,15.057736,-2.240142,6380
1,per_building,LIB,M0,12,target_cum_h12,P01_md3_lr003_mc5,12,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12743,1416,...,1999,8.202193,145.585695,196.250438,134.994728,14.457421,0.918501,19.301280,-4.843858,6376
2,per_building,LIB,M0,16,target_cum_h16,P01_md3_lr003_mc5,12,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12739,1416,...,1999,9.009242,211.977062,283.388836,196.043705,15.764805,0.902545,21.326058,-5.561253,6372
3,per_building,LIB,M0,20,target_cum_h20,P01_md3_lr003_mc5,12,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12736,1415,...,1999,9.310992,270.820706,362.317865,250.285483,16.116507,0.896606,21.709724,-5.593217,6368
4,per_building,LIB,M0,24,target_cum_h24,P01_md3_lr003_mc5,12,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12732,1415,...,1999,9.115497,318.245145,435.843306,298.230598,16.011401,0.895154,20.958142,-4.946741,6364
5,per_building,LIB,M0,36,target_cum_h36,P01_md3_lr003_mc5,12,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12721,1414,...,1999,8.776751,444.217394,641.123841,446.989835,16.018590,0.897770,22.199747,-6.181157,6352
6,per_building,LIB,M1,8,target_cum_h8,P01_md3_lr003_mc5,13,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12747,1416,...,1999,7.830992,97.891769,123.370843,79.599785,12.772892,0.929278,15.057736,-2.284844,6380
7,per_building,LIB,M1,12,target_cum_h12,P01_md3_lr003_mc5,13,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12743,1416,...,1999,8.332020,149.480643,195.735224,134.278455,14.380711,0.918929,19.301280,-4.920568,6376
8,per_building,LIB,M1,16,target_cum_h16,P01_md3_lr003_mc5,13,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12739,1416,...,1999,9.004107,214.168436,277.602957,192.594970,15.487475,0.906484,21.326058,-5.838583,6372
9,per_building,LIB,M1,20,target_cum_h20,P01_md3_lr003_mc5,13,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",12736,1415,...,1999,9.206853,269.563189,357.694051,247.810690,15.957149,0.899228,21.709724,-5.752575,6368


,scope,building_or_global,mode,horizon_h,target_name,preset_id,n_features,feature_list,n_train_rows,n_val_rows,...,best_iteration,val_wape,val_rmse,rmse,mae,wape_pct,r2,baseline_wape_pct,delta_wape_vs_baseline,n_test_eval_rows
0,global,GLOBAL,M0,8,target_cum_h8,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80586,8954,...,1999,10.639156,145.434554,145.854882,83.877194,14.041395,0.942624,18.424599,-4.383203,38164
1,global,GLOBAL,M0,12,target_cum_h12,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80564,8952,...,1999,12.055502,241.950972,242.455080,142.187447,15.879556,0.926774,24.233447,-8.353891,38140
2,global,GLOBAL,M0,16,target_cum_h16,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80543,8949,...,1999,12.760497,340.684286,332.890037,198.310135,16.624808,0.919865,26.905250,-10.280442,38116
3,global,GLOBAL,M0,20,target_cum_h20,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80521,8947,...,1999,13.040361,431.273065,412.834588,248.760211,16.696610,0.919454,26.939115,-10.242505,38092
4,global,GLOBAL,M0,24,target_cum_h24,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80500,8944,...,1999,12.141407,476.804019,480.099345,289.936464,16.224936,0.923560,25.532144,-9.307209,38068
5,global,GLOBAL,M0,36,target_cum_h36,P01_md3_lr003_mc5,40,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80435,8937,...,1999,11.908247,704.658604,690.823506,426.822255,15.935661,0.928913,27.043178,-11.107517,37996
6,global,GLOBAL,M1,8,target_cum_h8,P01_md3_lr003_mc5,41,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80586,8954,...,1999,10.834547,147.288900,144.931141,83.703373,14.012297,0.943349,18.424599,-4.412302,38164
7,global,GLOBAL,M1,12,target_cum_h12,P01_md3_lr003_mc5,41,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80564,8952,...,1999,12.221315,243.806235,241.987161,142.011005,15.859851,0.927056,24.233447,-8.373597,38140
8,global,GLOBAL,M1,16,target_cum_h16,P01_md3_lr003_mc5,41,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80543,8949,...,1999,12.962128,345.284013,331.373977,197.395606,16.548141,0.920594,26.905250,-10.357109,38116
9,global,GLOBAL,M1,20,target_cum_h20,P01_md3_lr003_mc5,41,"feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms...",80521,8947,...,1999,12.977054,428.998898,410.240425,246.591782,16.551067,0.920463,26.939115,-10.388049,38092


## Matched comparison vs LSTM and persistence baseline

The notebook writes two post-run summary artifacts:

- a compact XGBoost summary that keeps portfolio-mean and pooled-global views separate
- a matched XGBoost vs LSTM comparison where LSTM rows are available in the current thesis result folder

When an exact LSTM comparator is not available yet, the notebook leaves the LSTM fields empty rather than silently mixing unmatched comparisons.

In [9]:
if REBUILD_SUMMARIES and paths['per_building'].exists() and paths['global'].exists():
    summary_df, matched_df = build_summary_artifacts(config)
elif paths['summary'].exists() and paths['matched'].exists():
    summary_df = pd.read_csv(paths['summary'])
    matched_df = pd.read_csv(paths['matched'])
else:
    display(Markdown('> Summary artifacts are not available yet. Run the full matrix first.'))

if summary_df is not None:
    display(summary_df)

if matched_df is not None:
    display(matched_df)
    matched_available_df = matched_df.loc[matched_df['lstm_wape'].notna()].copy()
    if not matched_available_df.empty:
        display(matched_available_df.sort_values(['scope', 'horizon_h', 'mode', 'building']).reset_index(drop=True))

,scope,building_or_global,mode,horizon_h,target_name,preset_id,n_cases,mean_wape_pct,mean_rmse,mean_mae,mean_r2,mean_baseline_wape_pct,mean_delta_wape_vs_baseline
0,global,GLOBAL,M0,8,target_cum_h8,P01_md3_lr003_mc5,1,14.041395,145.854882,83.877194,0.942624,18.424599,-4.383203
1,global,GLOBAL,M1,8,target_cum_h8,P01_md3_lr003_mc5,1,14.012297,144.931141,83.703373,0.943349,18.424599,-4.412302
2,global,GLOBAL,M2,8,target_cum_h8,P01_md3_lr003_mc5,1,13.839667,140.015487,82.672155,0.947127,18.424599,-4.584932
3,global,GLOBAL,M3,8,target_cum_h8,P01_md3_lr003_mc5,1,13.753894,140.107611,82.159785,0.947057,18.424599,-4.670705
4,global,GLOBAL,M0,12,target_cum_h12,P01_md3_lr003_mc5,1,15.879556,242.455080,142.187447,0.926774,24.233447,-8.353891
5,global,GLOBAL,M1,12,target_cum_h12,P01_md3_lr003_mc5,1,15.859851,241.987161,142.011005,0.927056,24.233447,-8.373597
6,global,GLOBAL,M2,12,target_cum_h12,P01_md3_lr003_mc5,1,15.676712,237.171158,140.371161,0.929931,24.233447,-8.556735
7,global,GLOBAL,M3,12,target_cum_h12,P01_md3_lr003_mc5,1,15.366426,232.096955,137.592819,0.932897,24.233447,-8.867022
8,global,GLOBAL,M0,16,target_cum_h16,P01_md3_lr003_mc5,1,16.624808,332.890037,198.310135,0.919865,26.905250,-10.280442
9,global,GLOBAL,M1,16,target_cum_h16,P01_md3_lr003_mc5,1,16.548141,331.373977,197.395606,0.920594,26.905250,-10.357109


,scope,building,mode,horizon_h,xgb_wape,xgb_rmse,lstm_wape,lstm_rmse,delta_wape_xgb_minus_lstm
0,global,GLOBAL,M0,8,14.041395,145.854882,NaN,NaN,NaN
1,global,GLOBAL,M1,8,14.012297,144.931141,NaN,NaN,NaN
2,global,GLOBAL,M2,8,13.839667,140.015487,NaN,NaN,NaN
3,global,GLOBAL,M3,8,13.753894,140.107611,NaN,NaN,NaN
4,global,GLOBAL,M0,12,15.879556,242.455080,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
187,portfolio_mean,PORTFOLIO,M3,24,17.550182,457.357510,NaN,NaN,NaN
188,portfolio_mean,PORTFOLIO,M0,36,17.490503,665.947529,NaN,NaN,NaN
189,portfolio_mean,PORTFOLIO,M1,36,17.311858,665.310435,NaN,NaN,NaN
190,portfolio_mean,PORTFOLIO,M2,36,17.394611,661.328723,NaN,NaN,NaN


## Caveats and next step

Important caveats:

- this notebook locks the XGBoost **preset**, not final thesis interpretation
- `WAPE + RMSE` remain the headline pair; `MAPE` is deliberately not used for model ranking
- the pooled global track uses only safe static `stat_*` context, not target-derived training summaries
- exact matched LSTM rows do not yet exist for every horizon / scope combination in the current result folder

### What This Means For The Next Notebook

If the frozen XGBoost preset looks credible, the next notebook can focus on a narrower thesis comparison:

- strongest XGBoost mode(s) only
- strongest matched LSTM comparator(s)
- clean horizon-by-horizon interpretation
- optional SHAP only after the model family comparison is stable